"""
================================================================================
BERT Pre-training Loss 최적화 & 추론 검증 실험 (v2 - MLM 개선판)
================================================================================
 
[v1에서 MLM이 학습되지 않았던 근본 원인 3가지]
 
문제 A: 완전 랜덤 데이터 (가장 치명적)
  - np.random.randint(7, 5000)으로 생성한 토큰은 아무 패턴이 없음
  - BERT MLM은 "주변 문맥으로 마스크된 토큰을 예측"하는 task
  - 주변 토큰이 랜덤이면 → 문맥 정보 = 0 → 예측 불가능
  - ln(5000) ≈ 8.52이고, loss가 7.6까지 내렸다는건
    uniform(8.52)보다 나아진 것이지만, 5000개 중 top-1 맞출 확률은 ~0%
 
문제 B: vocab_size=5000이 데모에 너무 큼
  - 5000개 클래스에서 top-1 accuracy는 매우 작은 수치
  - 학습이 되고 있어도 차트에서는 0%로 보임
 
문제 C: 모델 크기 vs 데이터 크기 불균형
  - d_model=256, 6 layers = ~5M params
  - 2000개 랜덤 샘플로는 과적합도 어려움
 
[v2 해결 전략]
1. "학습 가능한 패턴"이 있는 합성 데이터 생성
   - Bigram 패턴: 특정 토큰 뒤에 특정 토큰이 오는 규칙
   - 문서 구조: 같은 문서의 문장은 유사한 토큰 분포
2. vocab_size를 500으로 축소 (데모 실험에 적합)
3. 모델 크기를 데이터에 맞게 조정
4. MLM loss에 가중치(weight) 부여로 학습 집중
================================================================================
"""


In [1]:
from __future__ import absolute_import, division, print_function, unicode_literals
 
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import math
import random
import copy
import json
from datetime import datetime


In [2]:
# ============================================================================
# 시드 고정
# ============================================================================
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device}")
 


PyTorch: 2.7.1+cu118 | Device: cuda


In [3]:
# ============================================================================
# [핵심 변경] Config - 데모 실험에 적합한 크기
# ============================================================================
"""
[회고] 왜 모델 크기를 줄였는가?
 
v1: d_model=256, n_layer=6, n_vocab=5000, 데이터 2000개
  → 파라미터 ~5M, 학습 가능한 패턴 없음 → 학습 불가
 
v2: d_model=128, n_layer=3, n_vocab=500, 데이터 5000개
  → 파라미터 ~800K, 패턴이 있는 데이터 → 학습 가능
 
원칙: 모델 복잡도 ∝ 데이터의 패턴 복잡도
- 랜덤 데이터에 큰 모델 = 학습 불가 (패턴이 없으므로)
- 패턴 데이터에 적절한 모델 = 학습 가능
- 실제 kowiki 데이터에는 큰 모델 사용
"""
 
class Config:
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)
 
config = Config({
    "d_model": 128,
    "n_head": 4,
    "d_head": 32,
    "dropout": 0.1,
    "d_ff": 512,
    "layernorm_epsilon": 1e-6,
    "n_layer": 3,
    "n_seq": 64,         # 짧은 시퀀스 (빠른 실험)
    "n_vocab": 500,      # 핵심: 작은 vocab으로 학습 가능성 확보
    "i_pad": 0
})
 
# 특수 토큰 ID
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
SEP_ID = 4
CLS_ID = 5
MASK_ID = 6
N_SPECIAL = 7  # 0~6은 특수 토큰
 


In [24]:
# ============================================================================
# 모델 구성 요소 (v1과 동일 - 검증된 부분)
# ============================================================================
 
def get_pad_mask(tokens, i_pad=0):
    return (tokens == i_pad).float().unsqueeze(1)
 
class SharedEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_vocab = config.n_vocab
        self.d_model = config.d_model
        self.shared_weights = nn.Parameter(torch.empty(self.n_vocab, self.d_model))
        nn.init.trunc_normal_(self.shared_weights, std=0.02)
 
    def forward(self, inputs, mode="embedding"):
        if mode == "embedding":
            inputs = torch.clamp(inputs, 0, self.n_vocab - 1)
            return self.shared_weights[inputs.long()]
        elif mode == "linear":
            bs, seq, _ = inputs.shape
            return torch.matmul(inputs.view(-1, self.d_model), self.shared_weights.T).view(bs, seq, self.n_vocab)
 
class PositionEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embedding = nn.Embedding(config.n_seq, config.d_model)
        nn.init.trunc_normal_(self.embedding.weight, std=0.02)
 
    def forward(self, inputs):
        pos = torch.arange(inputs.size(1), device=inputs.device).unsqueeze(0)
        return self.embedding(pos)
 
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.d_head = config.d_head
        self.W_Q = nn.Linear(config.d_model, config.n_head * config.d_head)
        self.W_K = nn.Linear(config.d_model, config.n_head * config.d_head)
        self.W_V = nn.Linear(config.d_model, config.n_head * config.d_head)
        self.W_O = nn.Linear(config.n_head * config.d_head, config.d_model)
 
    def forward(self, Q, K, V, mask):
        bs = Q.shape[0]
        q = self.W_Q(Q).view(bs, -1, self.n_head, self.d_head).transpose(1, 2)
        k = self.W_K(K).view(bs, -1, self.n_head, self.d_head).transpose(1, 2)
        v = self.W_V(V).view(bs, -1, self.n_head, self.d_head).transpose(1, 2)
 
        scale = math.sqrt(self.d_head)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        scores = scores - mask.unsqueeze(1) * 1e9
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(bs, -1, self.n_head * self.d_head)
        return self.W_O(out)
 
class EncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = MultiHeadAttention(config)
        self.norm1 = nn.LayerNorm(config.d_model, eps=config.layernorm_epsilon)
        self.ff1 = nn.Linear(config.d_model, config.d_ff)
        self.ff2 = nn.Linear(config.d_ff, config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model, eps=config.layernorm_epsilon)
        self.drop = nn.Dropout(config.dropout)
        self.gelu = nn.GELU()
 
    def forward(self, x, mask):
        h = self.norm1(x + self.drop(self.attn(x, x, x, mask)))
        return self.norm2(h + self.drop(self.ff2(self.gelu(self.ff1(h)))))
 
class BERT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.i_pad = config.i_pad
        self.embedding = SharedEmbedding(config)
        self.position = PositionEmbedding(config)
        self.segment = nn.Embedding(2, config.d_model)
        self.norm = nn.LayerNorm(config.d_model, eps=config.layernorm_epsilon)
        self.layers = nn.ModuleList([EncoderLayer(config) for _ in range(config.n_layer)])
        self.drop = nn.Dropout(config.dropout)
 
    def forward(self, tokens, segs):
        mask = get_pad_mask(tokens, self.i_pad)
        embed = self.embedding(tokens) + self.position(tokens) + self.segment(segs)
        h = self.drop(self.norm(embed))
        for layer in self.layers:
            h = layer(h, mask)
            
        # [수정 전]
        # cls_out = h[:, 0]  <-- 이미 여기서 자르고 있었습니다.
        # lm_out = self.embedding(h, mode="linear")
        # return cls_out, lm_out

        # [수정 후] 자르지 않고 전체 시퀀스인 'h'를 그대로 반환합니다.
        lm_out = self.embedding(h, mode="linear")
        return h, lm_out
 


In [33]:
# ============================================================================
# [핵심 수정] NSP Head - Softmax 완전 제거
# ============================================================================
class BertPooler(nn.Module):
    """[CLS] 토큰의 의미를 부드럽게 압축하는 전용 레이어"""
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.d_model, config.d_model)
        self.activation = nn.Tanh()

    def forward(self, hidden_states):
        # hidden_states 구조: (batch_size, sequence_length, hidden_size)
        first_token_tensor = hidden_states[:, 0] # [CLS] 토큰 추출
        return self.activation(self.dense(first_token_tensor))
class NSPHead(nn.Module):
    """Pooler를 거친 결과물을 2진 분류(0 or 1)로 변환"""
    def __init__(self, config):
        super().__init__()
        self.proj = nn.Linear(config.d_model, 2)

    def forward(self, pooled_output):
        return self.proj(pooled_output)  # 여전히 raw logits 반환!

class BERTPreTrainModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BERT(config)
        self.pooler = BertPooler(config) # Pooler 추가
        self.nsp_head = NSPHead(config)

    def forward(self, tokens, segs):
        # 1. BERT의 모든 토큰 출력값 (lm_logits 계산용)
        # (주의: 기존 BERT 클래스 안에서 cls_out을 분리하지 않고 전체 hidden_states를 반환하도록 BERT 클래스의 forward 마지막 줄도 수정이 필요할 수 있습니다.)
        hidden_states, lm_logits = self.bert(tokens.long(), segs.long()) 
        
        # 2. Pooler 통과 후 NSP 예측
        pooled_out = self.pooler(hidden_states)
        nsp_logits = self.nsp_head(pooled_out)
        
        return nsp_logits, lm_logits


In [17]:
# ============================================================================
# [핵심 변경] 학습 가능한 합성 데이터 생성
# ============================================================================
"""
[회고] 왜 v1의 랜덤 데이터로는 MLM이 학습될 수 없었는가?
 
BERT의 MLM은 본질적으로: P(masked_token | surrounding_context)
- 랜덤 데이터: P(token_i | context) = 1/V (uniform) → 학습 불가능
- 실제 언어: P("먹었다" | "밥을 ___") >> 1/V → 학습 가능
 
해결: Bigram 패턴을 가진 합성 데이터 생성
- 특정 토큰 뒤에 특정 토큰이 올 확률이 높도록 설계
- 모델이 "A 다음에는 B가 온다"는 패턴을 학습 가능
- 이것이 실제 언어 데이터의 n-gram 통계를 모사
 
또한 NSP도 의미있게:
- is_next=1: 같은 "문서"의 연속된 문장
- is_next=0: 다른 "문서"에서 가져온 문장
- 같은 문서는 유사한 토큰 분포를 가지므로 학습 가능
"""
 
class StructuredDataGenerator:
    """학습 가능한 패턴이 있는 합성 데이터 생성기"""
 
    def __init__(self, n_vocab, n_special=7, n_topics=20, n_patterns_per_topic=10):
        self.n_vocab = n_vocab
        self.n_special = n_special
        self.n_topics = n_topics
 
        # 1) 토픽별 토큰 분포 생성 (Zipf-like)
        self.topic_dists = []
        vocab_tokens = list(range(n_special, n_vocab))
        for t in range(n_topics):
            # 각 토픽마다 "자주 쓰는 토큰"이 다름
            weights = np.random.dirichlet(np.ones(len(vocab_tokens)) * 0.1)
            # 토픽별로 상위 30개 토큰에 가중치 집중
            top_indices = np.random.choice(len(vocab_tokens), 30, replace=False)
            weights[top_indices] *= 10
            weights /= weights.sum()
            self.topic_dists.append(weights)
 
        # 2) Bigram 전이 확률 (핵심: MLM이 학습할 패턴)
        #    "토큰 A 다음에 토큰 B가 자주 온다"
        self.bigram_probs = {}
        for t_id in vocab_tokens:
            # 각 토큰에 대해 3~8개의 "자주 따라오는" 토큰 설정
            n_followers = random.randint(3, 8)
            followers = random.sample(vocab_tokens, n_followers)
            probs = np.random.dirichlet(np.ones(n_followers) * 0.5)
            self.bigram_probs[t_id] = (followers, probs)
 
    def generate_sentence(self, topic_id, min_len=5, max_len=20):
        """토픽 기반 문장 생성 (bigram 패턴 포함)"""
        dist = self.topic_dists[topic_id]
        vocab_tokens = list(range(self.n_special, self.n_vocab))
        length = random.randint(min_len, max_len)
 
        tokens = []
        # 첫 토큰: 토픽 분포에서 샘플링
        first = np.random.choice(vocab_tokens, p=dist)
        tokens.append(first)
 
        for _ in range(length - 1):
            prev = tokens[-1]
            # 50% 확률로 bigram 패턴 따르기, 50% 토픽 분포
            if random.random() < 0.5 and prev in self.bigram_probs:
                followers, probs = self.bigram_probs[prev]
                tokens.append(np.random.choice(followers, p=probs))
            else:
                tokens.append(np.random.choice(vocab_tokens, p=dist))
 
        return tokens
 
    def generate_document(self, n_sentences=5):
        """한 토픽으로 구성된 문서 생성"""
        topic_id = random.randint(0, self.n_topics - 1)
        return [self.generate_sentence(topic_id) for _ in range(n_sentences)], topic_id
 
    def generate_pretrain_data(self, n_samples, n_seq):
        """BERT 사전학습 데이터 생성"""
        enc_tokens = np.zeros((n_samples, n_seq), dtype=np.int64)
        segments = np.zeros((n_samples, n_seq), dtype=np.int64)
        labels_nsp = np.zeros(n_samples, dtype=np.int64)
        labels_mlm = np.zeros((n_samples, n_seq), dtype=np.int64)
 
        max_seq = n_seq - 3  # [CLS], [SEP], [SEP]
 
        for i in range(n_samples):
            doc_a, topic_a = self.generate_document(n_sentences=3)
            doc_b, topic_b = self.generate_document(n_sentences=3)
 
            # Sentence A: doc_a의 첫 문장
            sent_a = doc_a[0]
 
            # NSP: 50% 실제 다음 문장, 50% 랜덤 문서의 문장
            if random.random() < 0.5:
                sent_b = doc_a[1]  # 실제 다음 문장 (같은 토픽)
                labels_nsp[i] = 1
            else:
                sent_b = doc_b[random.randint(0, len(doc_b)-1)]  # 다른 문서
                labels_nsp[i] = 0
 
            # 길이 조정
            half = max_seq // 2
            sent_a = sent_a[:half]
            sent_b = sent_b[:max_seq - len(sent_a)]
 
            # [CLS] sent_a [SEP] sent_b [SEP]
            seq = [CLS_ID] + sent_a + [SEP_ID] + sent_b + [SEP_ID]
            seg = [0] * (len(sent_a) + 2) + [1] * (len(sent_b) + 1)
 
            # Padding
            pad_len = n_seq - len(seq)
            seq = seq + [PAD_ID] * pad_len
            seg = seg + [0] * pad_len
 
            # MLM Masking (15%)
            original_seq = seq.copy()
            n_mask = max(1, int(len(sent_a + sent_b) * 0.15))
            # 마스크 가능한 위치: 특수 토큰/패딩 제외
            mask_candidates = [j for j in range(len(seq))
                               if seq[j] not in [PAD_ID, CLS_ID, SEP_ID, MASK_ID]]
            random.shuffle(mask_candidates)
 
            mlm_label = np.zeros(n_seq, dtype=np.int64)
            for j in mask_candidates[:n_mask]:
                mlm_label[j] = original_seq[j]  # 원래 토큰 저장
                dice = random.random()
                if dice < 0.8:
                    seq[j] = MASK_ID
                elif dice < 0.9:
                    pass  # keep original
                else:
                    seq[j] = random.randint(N_SPECIAL, self.n_vocab - 1)
 
            enc_tokens[i] = seq[:n_seq]
            segments[i] = seg[:n_seq]
            labels_mlm[i] = mlm_label
 
        return enc_tokens, segments, labels_nsp, labels_mlm
 


In [18]:
# ============================================================================
# 학습 보조 함수
# ============================================================================
 
class CosineScheduleWithWarmup:
    """
    [v2 개선] warmup 비율을 전체의 10%로 설정하되,
    minimum lr을 0이 아닌 max_lr의 5%로 유지
    → v1에서 lr이 0에 수렴하여 후반 학습이 멈추는 문제 해결
    """
    def __init__(self, optimizer, total_steps, warmup_steps, max_lr=5e-4, min_lr_ratio=0.05):
        self.optimizer = optimizer
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.max_lr = max_lr
        self.min_lr = max_lr * min_lr_ratio
        self.step_count = 0
 
    def step(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            lr = (self.step_count / max(1, self.warmup_steps)) * self.max_lr
        else:
            progress = (self.step_count - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        return lr
 
 
def compute_mlm_accuracy(logits, labels):
    """마스크 위치에서만 정확도 계산"""
    mask = (labels != 0)
    if mask.sum() == 0:
        return torch.tensor(0.0)
    preds = logits.argmax(dim=-1)
    return ((preds == labels) & mask).float().sum() / mask.float().sum()
 
def compute_nsp_accuracy(logits, labels):
    return (logits.argmax(dim=-1) == labels).float().mean()


In [46]:
# ============================================================================
# [핵심 변경] 학습 함수 - MLM loss 가중치 + 상세 모니터링
# ============================================================================
"""
[회고] MLM loss 가중치가 필요한 이유
 
BERT에서 total_loss = loss_nsp + loss_mlm
- NSP loss: 2-class 분류 → 최대 ln(2) ≈ 0.693
- MLM loss: V-class 분류 → 최대 ln(V) ≈ 6.2 (V=500)
- MLM loss가 자연스럽게 더 크므로 gradient도 MLM 위주
- 하지만 마스크 위치가 전체의 15%만이므로 유효 gradient 적음
 
해결: mlm_weight를 명시적으로 설정하여 MLM 학습 강화
"""
 
def train_model(config, data_tuple, epochs=60, batch_size=128,
                max_lr=5e-4, mlm_weight=1.0, label="Improved"):
    """통합 학습 함수"""
    enc_tokens, segments, labels_nsp, labels_mlm = data_tuple
 
    model = BERTPreTrainModel(config).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*70}")
    print(f"[{label}] Training Start")
    print(f"  Params: {n_params:,} | Data: {len(enc_tokens)} | Epochs: {epochs}")
    print(f"  Vocab: {config.n_vocab} | d_model: {config.d_model} | layers: {config.n_layer}")
    print(f"  MLM weight: {mlm_weight} | Max LR: {max_lr}")
    print(f"{'='*70}")
 
    loss_fn_nsp = nn.CrossEntropyLoss()
    loss_fn_mlm = nn.CrossEntropyLoss(ignore_index=0)  # 패딩 무시!
 
    optimizer = optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.01,
                            betas=(0.9, 0.999), eps=1e-8)
 
    total_steps = math.ceil(len(enc_tokens) / batch_size) * epochs
    warmup_steps = total_steps // 10
    scheduler = CosineScheduleWithWarmup(optimizer, total_steps, warmup_steps,
                                         max_lr=max_lr, min_lr_ratio=0.05)
 
    dataset = TensorDataset(
        torch.tensor(enc_tokens, dtype=torch.long).to(device),
        torch.tensor(segments, dtype=torch.long).to(device),
        torch.tensor(labels_nsp, dtype=torch.long).to(device),
        torch.tensor(labels_mlm, dtype=torch.long).to(device),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)
 
    history = {
        'nsp_loss': [], 'mlm_loss': [], 'total_loss': [],
        'nsp_acc': [], 'mlm_acc': [], 'lr': [],
        'mlm_top5_acc': [],  # Top-5 정확도 추가
    }
 
    for epoch in range(epochs):
        model.train()
        ep = {k: 0.0 for k in ['nsp_loss', 'mlm_loss', 'nsp_acc', 'mlm_acc', 'top5']}
        n_batches = 0
 
        for batch in loader:
            tok, seg, lbl_nsp, lbl_mlm = batch
            optimizer.zero_grad()
 
            nsp_logits, mlm_logits = model(tok, seg)
 
            loss_nsp = loss_fn_nsp(nsp_logits, lbl_nsp.long())
            loss_mlm = loss_fn_mlm(
                mlm_logits.view(-1, config.n_vocab),
                lbl_mlm.long().view(-1)
            )
 
            # MLM 가중치 적용
            total_loss = loss_nsp + mlm_weight * loss_mlm
            total_loss.backward()
 
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            lr = scheduler.step()
 
            # Metrics
            with torch.no_grad():
                ep['nsp_loss'] += loss_nsp.item()
                ep['mlm_loss'] += loss_mlm.item()
                ep['nsp_acc'] += compute_nsp_accuracy(nsp_logits, lbl_nsp).item()
                ep['mlm_acc'] += compute_mlm_accuracy(mlm_logits, lbl_mlm).item()
 
                # Top-5 accuracy
                mask = (lbl_mlm != 0)
                if mask.sum() > 0:
                    top5 = mlm_logits.topk(5, dim=-1).indices
                    hits = (top5 == lbl_mlm.unsqueeze(-1)).any(dim=-1) & mask
                    ep['top5'] += (hits.float().sum() / mask.float().sum()).item()
 
            n_batches += 1
 
        # Record history
        for k in ['nsp_loss', 'mlm_loss', 'nsp_acc', 'mlm_acc']:
            history[k].append(ep[k] / n_batches)
        history['total_loss'].append((ep['nsp_loss'] + ep['mlm_loss']) / n_batches)
        history['mlm_top5_acc'].append(ep['top5'] / n_batches)
        history['lr'].append(lr)
 
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:3d}/{epochs} | "
                  f"NSP L:{history['nsp_loss'][-1]:.4f} A:{history['nsp_acc'][-1]:.3f} | "
                  f"MLM L:{history['mlm_loss'][-1]:.4f} A:{history['mlm_acc'][-1]:.4f} "
                  f"Top5:{history['mlm_top5_acc'][-1]:.4f} | "
                  f"LR:{lr:.6f}")
 
    return history, model


In [48]:
# ============================================================================
# 원본 방식 학습 (비교용 - 문제점 포함)
# ============================================================================
class PooledOutput_ORIGINAL(nn.Module):
    """문제: softmax 포함"""
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.d_model, config.d_model)
        self.proj = nn.Linear(config.d_model, 2)
    def forward(self, x):
        return F.softmax(self.proj(torch.tanh(self.dense(x))), dim=-1)  # softmax!
 
class OriginalPreTrainModel(nn.Module):
    """원본 모델: double softmax + raw CE loss"""
    def __init__(self, config):
        super().__init__()
        self.bert = BERT(config)
        self.nsp_head = PooledOutput_ORIGINAL(config)
        
    def forward(self, tokens, segs):
        # 이제 cls_out 변수에는 잘리지 않은 전체 시퀀스(h)가 들어옵니다.
        cls_out, lm_logits = self.bert(tokens.long(), segs.long())
        
        # [핵심 추가 사항] 분류를 위해 첫 번째 [CLS] 토큰만 명시적으로 잘라냅니다!
        cls_out = cls_out[:, 0] 
        
        nsp_out = F.softmax(self.nsp_head(cls_out), dim=-1)  # double softmax!
        mlm_out = F.softmax(lm_logits, dim=-1)  # softmax on MLM too!
        return nsp_out, mlm_out
 
def train_original(config, data_tuple, epochs=60, batch_size=128):
    """원본 방식 학습 (모든 문제점 포함)"""
    enc_tokens, segments, labels_nsp, labels_mlm = data_tuple
 
    model = OriginalPreTrainModel(config).to(device)
    print(f"\n{'='*70}")
    print(f"[Original] Training Start (with all issues)")
    print(f"  Issues: Double softmax + No ignore_index + No scheduler + No clip")
    print(f"{'='*70}")
 
    loss_fn_nsp = nn.CrossEntropyLoss()
    loss_fn_mlm = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)  
    optimizer = optim.Adam(model.parameters(), lr=1e-4)  # 고정 lr!
 
    dataset = TensorDataset(
        torch.tensor(enc_tokens, dtype=torch.long).to(device),
        torch.tensor(segments, dtype=torch.long).to(device),
        torch.tensor(labels_nsp, dtype=torch.long).to(device),
        torch.tensor(labels_mlm, dtype=torch.long).to(device),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)
 
    history = {
        'nsp_loss': [], 'mlm_loss': [], 'total_loss': [],
        'nsp_acc': [], 'mlm_acc': [], 'lr': [],
        'mlm_top5_acc': [],
    }
 
    for epoch in range(epochs):
        model.train()
        ep = {k: 0.0 for k in ['nsp_loss', 'mlm_loss', 'nsp_acc', 'mlm_acc', 'top5']}
        n_batches = 0
 
        for batch in loader:
            tok, seg, lbl_nsp, lbl_mlm = batch
            optimizer.zero_grad()
 
            nsp_out, mlm_out = model(tok, seg)
 
            loss_nsp = loss_fn_nsp(nsp_out, lbl_nsp.long())
            loss_mlm = loss_fn_mlm(mlm_out.view(-1, config.n_vocab), lbl_mlm.long().view(-1))
 
            total_loss = loss_nsp + loss_mlm
            total_loss.backward()
            # gradient clipping 없음!
            optimizer.step()
 
            with torch.no_grad():
                ep['nsp_loss'] += loss_nsp.item()
                ep['mlm_loss'] += loss_mlm.item()
 
                # 원본식 accuracy (부풀려짐)
                ep['nsp_acc'] += (nsp_out.argmax(-1) == lbl_nsp).float().mean().item()
                ep['mlm_acc'] += (mlm_out.argmax(-1) == lbl_mlm).float().mean().item()
 
                mask = (lbl_mlm != 0)
                if mask.sum() > 0:
                    top5 = mlm_out.topk(5, dim=-1).indices
                    hits = (top5 == lbl_mlm.unsqueeze(-1)).any(dim=-1) & mask
                    ep['top5'] += (hits.float().sum() / mask.float().sum()).item()
 
            n_batches += 1
 
        for k in ['nsp_loss', 'mlm_loss', 'nsp_acc', 'mlm_acc']:
            history[k].append(ep[k] / n_batches)
        history['total_loss'].append((ep['nsp_loss'] + ep['mlm_loss']) / n_batches)
        history['mlm_top5_acc'].append(ep['top5'] / n_batches)
        history['lr'].append(1e-4)
 
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:3d}/{epochs} | "
                  f"NSP L:{history['nsp_loss'][-1]:.4f} A:{history['nsp_acc'][-1]:.3f} | "
                  f"MLM L:{history['mlm_loss'][-1]:.4f} A:{history['mlm_acc'][-1]:.4f} "
                  f"Top5:{history['mlm_top5_acc'][-1]:.4f}")
 
    return history, model


In [42]:
# ============================================================================
# 추론 검증 (Inference Validation)
# ============================================================================
def inference_validation(model, config, data_generator, n_samples=1000, label="Model"):
    """학습되지 않은 새 데이터로 추론 능력 검증"""
    print(f"\n{'='*70}")
    print(f"[{label}] Inference Validation (unseen {n_samples} samples)")
    print(f"{'='*70}")
 
    # 새로운 검증 데이터 생성
    test_data = data_generator.generate_pretrain_data(n_samples, config.n_seq)
    enc_tokens, segments, labels_nsp, labels_mlm = test_data
 
    dataset = TensorDataset(
        torch.tensor(enc_tokens, dtype=torch.long).to(device),
        torch.tensor(segments, dtype=torch.long).to(device),
        torch.tensor(labels_nsp, dtype=torch.long).to(device),
        torch.tensor(labels_mlm, dtype=torch.long).to(device),
    )
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
 
    model.eval()
    metrics = {k: 0.0 for k in ['nsp_correct', 'nsp_total',
                                  'mlm_correct', 'mlm_total',
                                  'top5_hits', 'top10_hits',
                                  'perp_sum', 'perp_count']}
    all_losses = []
 
    with torch.no_grad():
        for batch in loader:
            tok, seg, lbl_nsp, lbl_mlm = batch
            nsp_logits, mlm_logits = model(tok, seg)
 
            # NSP
            metrics['nsp_correct'] += (nsp_logits.argmax(-1) == lbl_nsp).sum().item()
            metrics['nsp_total'] += len(lbl_nsp)
 
            # MLM (마스크 위치만)
            mask = (lbl_mlm != 0)
            if mask.sum() == 0:
                continue
 
            preds = mlm_logits.argmax(-1)
            metrics['mlm_correct'] += ((preds == lbl_mlm) & mask).sum().item()
            metrics['mlm_total'] += mask.sum().item()
 
            # Top-K
            for k, key in [(5, 'top5_hits'), (10, 'top10_hits')]:
                topk = mlm_logits.topk(k, dim=-1).indices
                hits = (topk == lbl_mlm.unsqueeze(-1)).any(-1) & mask
                metrics[key] += hits.sum().item()
 
            # Perplexity
            log_p = F.log_softmax(mlm_logits, dim=-1)
            gathered = log_p.gather(2, lbl_mlm.long().unsqueeze(-1)).squeeze(-1)
            metrics['perp_sum'] += (-gathered[mask]).sum().item()
            metrics['perp_count'] += mask.sum().item()
 
            # Loss distribution
            per_loss = F.cross_entropy(
                mlm_logits.view(-1, config.n_vocab),
                lbl_mlm.long().view(-1), reduction='none'
            ).view_as(lbl_mlm)
            all_losses.extend(per_loss[mask].cpu().numpy().tolist())
 
    # 계산
    nsp_acc = metrics['nsp_correct'] / max(1, metrics['nsp_total'])
    mlm_acc = metrics['mlm_correct'] / max(1, metrics['mlm_total'])
    top5 = metrics['top5_hits'] / max(1, metrics['mlm_total'])
    top10 = metrics['top10_hits'] / max(1, metrics['mlm_total'])
    ppl = math.exp(metrics['perp_sum'] / max(1, metrics['perp_count']))
 
    # Random baselines
    rand_mlm = 1.0 / config.n_vocab
    rand_top5 = 5.0 / config.n_vocab
    rand_top10 = 10.0 / config.n_vocab
    rand_ppl = config.n_vocab
 
    print(f"\n  {'Metric':<25} {'Model':>10} {'Random':>10} {'Improvement':>12}")
    print(f"  {'-'*60}")
    print(f"  {'NSP Accuracy':<25} {nsp_acc:>10.4f} {'0.5000':>10} {nsp_acc/0.5:>10.2f}x")
    print(f"  {'MLM Accuracy (Top-1)':<25} {mlm_acc:>10.4f} {rand_mlm:>10.4f} {mlm_acc/rand_mlm:>10.1f}x")
    print(f"  {'MLM Top-5 Recall':<25} {top5:>10.4f} {rand_top5:>10.4f} {top5/rand_top5:>10.1f}x")
    print(f"  {'MLM Top-10 Recall':<25} {top10:>10.4f} {rand_top10:>10.4f} {top10/rand_top10:>10.1f}x")
    print(f"  {'Perplexity':<25} {ppl:>10.1f} {rand_ppl:>10.1f} {rand_ppl/ppl:>10.1f}x better")
 
    # 정당성 판단
    checks = [
        ("MLM Acc > 5x Random", mlm_acc > rand_mlm * 5),
        ("NSP Acc > 55%", nsp_acc > 0.55),
        ("Top-5 Recall > 5x Random", top5 > rand_top5 * 5),
        ("Perplexity < V/2", ppl < config.n_vocab / 2),
    ]
    print(f"\n  Validation Checks:")
    all_pass = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        if not passed: all_pass = False
        print(f"    [{status}] {name}")
 
    if all_pass:
        print(f"\n  >>> ALL PASSED: Model demonstrates meaningful learning!")
    else:
        print(f"\n  >>> SOME FAILED: Needs more training or tuning")
 
    return {
        'nsp_acc': nsp_acc, 'mlm_acc': mlm_acc,
        'top5': top5, 'top10': top10,
        'perplexity': ppl, 'losses': all_losses
    }


In [43]:
# ============================================================================
# 시각화
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
 
def plot_full_comparison(h_orig, h_impr, m_orig, m_impr, save_path):
    fig, axes = plt.subplots(2, 4, figsize=(24, 10))
    fig.suptitle('BERT Pre-training: Original vs Improved (v2)', fontsize=18, fontweight='bold')
 
    epochs_o = range(1, len(h_orig['nsp_loss']) + 1)
    epochs_i = range(1, len(h_impr['nsp_loss']) + 1)
 
    # Row 1: Loss & Accuracy
    # 1. NSP Loss
    ax = axes[0, 0]
    ax.plot(epochs_o, h_orig['nsp_loss'], 'r--', alpha=0.7, lw=1.5, label='Original')
    ax.plot(epochs_i, h_impr['nsp_loss'], 'b-', lw=2, label='Improved')
    ax.axhline(0.693, color='gray', ls=':', alpha=0.5, label='Random')
    ax.set_title('NSP Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
 
    # 2. MLM Loss (핵심!)
    ax = axes[0, 1]
    ax.plot(epochs_o, h_orig['mlm_loss'], 'r--', alpha=0.7, lw=1.5, label='Original')
    ax.plot(epochs_i, h_impr['mlm_loss'], 'b-', lw=2, label='Improved')
    ax.axhline(math.log(config.n_vocab), color='gray', ls=':', alpha=0.5, label=f'Random (ln{config.n_vocab})')
    ax.set_title('MLM Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
 
    # 3. NSP Accuracy
    ax = axes[0, 2]
    ax.plot(epochs_o, h_orig['nsp_acc'], 'r--', alpha=0.7, lw=1.5, label='Original')
    ax.plot(epochs_i, h_impr['nsp_acc'], 'b-', lw=2, label='Improved')
    ax.axhline(0.5, color='gray', ls=':', alpha=0.5)
    ax.set_title('NSP Accuracy', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
 
    # 4. MLM Accuracy (핵심!)
    ax = axes[0, 3]
    ax.plot(epochs_i, h_impr['mlm_acc'], 'b-', lw=2, label='Improved (true)')
    ax.axhline(1/config.n_vocab, color='gray', ls=':', alpha=0.5, label=f'Random (1/{config.n_vocab})')
    ax.set_title('MLM Top-1 Accuracy (masked only)', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
 
    # Row 2: Detailed metrics
    # 5. MLM Top-5 Accuracy
    ax = axes[1, 0]
    ax.plot(epochs_o, h_orig['mlm_top5_acc'], 'r--', alpha=0.7, lw=1.5, label='Original')
    ax.plot(epochs_i, h_impr['mlm_top5_acc'], 'b-', lw=2, label='Improved')
    ax.axhline(5/config.n_vocab, color='gray', ls=':', alpha=0.5, label='Random')
    ax.set_title('MLM Top-5 Accuracy', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
 
    # 6. Learning Rate
    ax = axes[1, 1]
    ax.plot(h_impr['lr'], 'g-', lw=2, label='Improved')
    ax.plot(h_orig['lr'], 'r--', alpha=0.5, lw=1, label='Original (fixed)')
    ax.set_title('Learning Rate', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
 
    # 7. Inference Metrics Bar Chart
    ax = axes[1, 2]
    cats = ['MLM\nTop-1', 'MLM\nTop-5', 'MLM\nTop-10', 'NSP\nAcc']
    orig_v = [m_orig['mlm_acc'], m_orig['top5'], m_orig['top10'], m_orig['nsp_acc']]
    impr_v = [m_impr['mlm_acc'], m_impr['top5'], m_impr['top10'], m_impr['nsp_acc']]
    x = np.arange(len(cats))
    w = 0.35
    b1 = ax.bar(x - w/2, orig_v, w, color='#e74c3c', alpha=0.8, label='Original')
    b2 = ax.bar(x + w/2, impr_v, w, color='#2980b9', alpha=0.8, label='Improved')
    for b in b1: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{b.get_height():.3f}', ha='center', fontsize=8)
    for b in b2: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{b.get_height():.3f}', ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(cats)
    ax.set_title('Inference Metrics', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3, axis='y')
 
    # 8. Perplexity Comparison
    ax = axes[1, 3]
    bars = ax.bar(['Original', 'Improved', 'Random'],
                  [m_orig['perplexity'], m_impr['perplexity'], config.n_vocab],
                  color=['#e74c3c', '#2980b9', '#95a5a6'], alpha=0.8)
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+5,
                f'{b.get_height():.0f}', ha='center', fontweight='bold')
    ax.set_title('MLM Perplexity (lower = better)', fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
 
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"\nChart saved: {save_path}")


In [49]:
# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print("BERT Pre-training Loss Optimization & Inference Validation (v2)")
    print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)
 
    # ---- 데이터 생성 ----
    print("\n[1/5] Generating structured synthetic data...")
    gen = StructuredDataGenerator(n_vocab=config.n_vocab, n_topics=20)
    train_data = gen.generate_pretrain_data(n_samples=5000, n_seq=config.n_seq)
 
    enc_t, seg_t, nsp_t, mlm_t = train_data
    n_masked = (mlm_t != 0).sum()
    print(f"  Samples: {len(enc_t)} | Seq len: {config.n_seq}")
    print(f"  Total masked tokens: {n_masked} ({n_masked/len(enc_t):.1f}/sample)")
    print(f"  NSP distribution: 0={np.sum(nsp_t==0)}, 1={np.sum(nsp_t==1)}")
    print(f"  Vocab: {config.n_vocab} (random baseline: {1/config.n_vocab:.4f})")
 
    EPOCHS = 120
 
    # ---- 원본 방식 ----
    print(f"\n[2/5] Training Original model ({EPOCHS} epochs)...")
    h_orig, model_orig = train_original(config, train_data, epochs=EPOCHS, batch_size=128)
 
    # ---- 개선 방식 ----
    print(f"\n[3/5] Training Improved model ({EPOCHS} epochs)...")
    h_impr, model_impr = train_model(config, train_data, epochs=EPOCHS, batch_size=128,
                                      max_lr=5e-4, mlm_weight=2.0, label="Improved")
 
    # ---- 추론 검증 ----
    print(f"\n[4/5] Inference Validation (unseen data)...")
    m_orig = inference_validation(model_orig, config, gen, n_samples=1000, label="Original")
    m_impr = inference_validation(model_impr, config, gen, n_samples=1000, label="Improved")
 
    # ---- 시각화 ----
    print(f"\n[5/5] Generating comparison chart...")
    chart_path = 'comparison_chart_v2.png'
    plot_full_comparison(h_orig, h_impr, m_orig, m_impr, chart_path)
 
    # ---- 최종 보고 ----
    print(f"""
{'='*70}
                    최종 실험 요약 보고 (v2)
{'='*70}
 
1. v1에서 MLM이 학습되지 않았던 근본 원인:
   A) 완전 랜덤 데이터 → 문맥으로 예측할 패턴 자체가 없었음
   B) vocab=5000이 데모에 너무 큼 → top-1 정확도가 항상 ~0%
   C) 모델(5M params) vs 데이터(2K random) 불균형
 
2. v2 개선 사항:
   A) Bigram 패턴 + 토픽 분포를 가진 합성 데이터
   B) vocab=500으로 축소 (데모 실험에 적합)
   C) 모델 크기 적정화 (d_model=128, 3 layers)
   D) LR scheduler의 min_lr 유지 (0에 수렴 방지)
   E) MLM loss 가중치 2.0 적용
 
3. 결과:
   Original NSP Acc:  {m_orig['nsp_acc']:.4f}  |  Improved: {m_impr['nsp_acc']:.4f}
   Original MLM Acc:  {m_orig['mlm_acc']:.4f}  |  Improved: {m_impr['mlm_acc']:.4f}
   Original Top-5:    {m_orig['top5']:.4f}  |  Improved: {m_impr['top5']:.4f}
   Original PPL:      {m_orig['perplexity']:.0f}     |  Improved: {m_impr['perplexity']:.0f}
 
4. 핵심 교훈:
   - Loss 개선 기법(softmax 제거, ignore_index 등)은 필요조건
   - 데이터에 학습 가능한 패턴이 있어야 충분조건 달성
   - 실제 kowiki 데이터 사용 시 더 극적인 개선 기대
{'='*70}
""")


BERT Pre-training Loss Optimization & Inference Validation (v2)
Time: 2026-03-13 09:15:47

[1/5] Generating structured synthetic data...
  Samples: 5000 | Seq len: 64
  Total masked tokens: 16406 (3.3/sample)
  NSP distribution: 0=2615, 1=2385
  Vocab: 500 (random baseline: 0.0020)

[2/5] Training Original model (120 epochs)...

[Original] Training Start (with all issues)
  Issues: Double softmax + No ignore_index + No scheduler + No clip
  Ep   1/120 | NSP L:0.6929 A:0.521 | MLM L:6.2145 A:0.0232 Top5:0.0183
  Ep  10/120 | NSP L:0.6925 A:0.509 | MLM L:6.2100 A:0.0005 Top5:0.0413
  Ep  20/120 | NSP L:0.6922 A:0.509 | MLM L:6.2087 A:0.0005 Top5:0.0407
  Ep  30/120 | NSP L:0.6927 A:0.520 | MLM L:6.2085 A:0.0005 Top5:0.0407
  Ep  40/120 | NSP L:0.6927 A:0.514 | MLM L:6.2086 A:0.0005 Top5:0.0407
  Ep  50/120 | NSP L:0.6919 A:0.525 | MLM L:6.2082 A:0.0005 Top5:0.0425
  Ep  60/120 | NSP L:0.6916 A:0.531 | MLM L:6.2078 A:0.0005 Top5:0.0437
  Ep  70/120 | NSP L:0.6924 A:0.517 | MLM L:6.2065 A: